# Data Preparation Notebook for Tableau Dashboard

This notebook prepares and cleans data for visualization in the Tableau dashboard.

In [2]:
import pandas as pd

# Convert Parquet Files to CSV Format

In [3]:
# convert a parquet file to csv file
def convert_parquet_to_csv(subdir, input_filename, output_filename):
    """
    Convert a Parquet file to a CSV file.

    Args:
        input_file (str): Path to the input Parquet file.
        output_file (str): Path to the output CSV file.
    """
    df = pd.read_parquet(f"data/{subdir}/"+input_filename+".parquet")
    df.to_csv(f"data/csv_files/{subdir}/"+output_filename+".csv", index=False)
    print(f"Converted {input_filename} to {output_filename}")

In [6]:
healthcare_access_files = ['can_healthcare_access', 'healthcare_access', 'healthcare_access_2017']
for file in healthcare_access_files:
    convert_parquet_to_csv('healthcare-access', file, file)

Converted can_healthcare_access to can_healthcare_access
Converted healthcare_access to healthcare_access
Converted healthcare_access_2017 to healthcare_access_2017


In [10]:
health_indicators_files = ['can_health_indicator', 'health_indicator24', 'can_health_indicator_health_provider']
for file in health_indicators_files:
    convert_parquet_to_csv('health-indicators', file, file)

Converted can_health_indicator to can_health_indicator
Converted health_indicator24 to health_indicator24
Converted can_health_indicator_health_provider to can_health_indicator_health_provider


# Check if Converted Files are Non-Empty

In [3]:
# check if converted files are non-empty
def check_non_empty_csv(subdir, filename):
    """
    Check if a CSV file is non-empty.

    Args:
        subdir (str): Subdirectory where the CSV file is located.
        filename (str): Name of the CSV file (without extension).
    
    Returns:
        bool: True if the file is non-empty, False otherwise.
    """
    df = pd.read_csv(f"data/csv_files/{subdir}/"+filename+".csv")
    print(f"Checking if {filename} is non-empty: {not df.empty}")
    print(f"Shape of {filename}: {df.shape}")

In [12]:
for file in healthcare_access_files:
    check_non_empty_csv('healthcare-access', file)

Checking if can_healthcare_access is non-empty: True
Shape of can_healthcare_access: (719, 7)
Checking if healthcare_access is non-empty: True
Shape of healthcare_access: (510, 18)
Checking if healthcare_access_2017 is non-empty: True
Shape of healthcare_access_2017: (46142, 10)


In [13]:
for file in health_indicators_files:
    check_non_empty_csv('health-indicators', file)

Checking if can_health_indicator is non-empty: True
Shape of can_health_indicator: (37827, 17)
Checking if health_indicator24 is non-empty: True
Shape of health_indicator24: (36970, 8)
Checking if can_health_indicator_health_provider is non-empty: True
Shape of can_health_indicator_health_provider: (16854, 23)


# Final Data Cleaning and Preparation

## Healthcare Access Datasets

In [5]:
can_hc_df = pd.read_csv("data/csv_files/healthcare-access/can_healthcare_access.csv")
can_hc_df.head()

,ref_date,geo,gender,unit,value,location,age
0,2018,Canada,Both sexes,Number_Thousands,1536.0,Canada,NaN
1,2018,Canada,Both sexes,Percent,5.1,Canada,NaN
2,2018,Canada,Males,Number_Thousands,683.0,Canada,NaN
3,2018,Canada,Males,Percent,4.6,Canada,NaN
4,2018,Canada,Females,Number_Thousands,853.0,Canada,NaN


In [9]:
hc_df = pd.read_csv("data/csv_files/healthcare-access/healthcare_access.csv")
hc_df.head()

,indigenous_group,gender,healthcare_access_experience,value,unmet_healthcare_need_past_12mo,consulted_provider_nonurgent_12mo,wait_time,wait_satisfaction,mental_health_status,needed_mental_health_care,mental_healthcare_needs_met,prescription_12mo,cost_related_nonadherence,unable_to_fill_prescription,traveled_for_care,reported_discrimination,importance_indigenous_support,reason_indigenous_support
0,First Nations,"Total, gender","Total, unmet health care needs in the past 12 ...",100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,First Nations,Men+,"Total, unmet health care needs in the past 12 ...",100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,First Nations,Women+,"Total, unmet health care needs in the past 12 ...",100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Métis,"Total, gender","Total, unmet health care needs in the past 12 ...",100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Métis,Men+,"Total, unmet health care needs in the past 12 ...",100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Drop "Total" Values

In [34]:
# check if any string column contains "Total"
def contains_total(df):
    string_cols = [col for col in df.columns if df[col].dtype == 'object']

    print(f"Checking for 'Total' in columns:")
    for col in string_cols:
        if df[col].str.contains("Total").any():
            print(f"Column '{col}' contains 'Total' ❌")
        else:
            print(f"Column '{col}' does not contain 'Total' ✅")

In [50]:
# drop any row where either column contains "Total"
def drop_total_rows(df):
    string_cols = [col for col in df.columns if df[col].dtype == 'object']
    total_mask = df[string_cols].apply(lambda x: x.str.contains("Total")).any(axis=1)
    return df[~total_mask]

### Healthcare Access

> Canadian Healthcare Access

In [56]:
can_hc_ft_df = drop_total_rows(can_hc_df)
print(can_hc_df.shape, can_hc_ft_df.shape)
can_hc_ft_df.head()

(719, 7) (719, 7)


,ref_date,geo,gender,unit,value,location,age
0,2018,Canada,Both sexes,Number_Thousands,1536.0,Canada,NaN
1,2018,Canada,Both sexes,Percent,5.1,Canada,NaN
2,2018,Canada,Males,Number_Thousands,683.0,Canada,NaN
3,2018,Canada,Males,Percent,4.6,Canada,NaN
4,2018,Canada,Females,Number_Thousands,853.0,Canada,NaN


In [57]:
contains_total(can_hc_ft_df)

Checking for 'Total' in columns:
Column 'geo' does not contain 'Total' ✅
Column 'gender' does not contain 'Total' ✅
Column 'unit' does not contain 'Total' ✅
Column 'location' does not contain 'Total' ✅
Column 'age' does not contain 'Total' ✅


> Indigenous Healthcare Access

In [36]:
contains_total(hc_df)

Checking for 'Total' in columns:
Column 'indigenous_group' does not contain 'Total' ✅
Column 'gender' contains 'Total' ❌
Column 'healthcare_access_experience' contains 'Total' ❌
Column 'unmet_healthcare_need_past_12mo' does not contain 'Total' ✅
Column 'consulted_provider_nonurgent_12mo' does not contain 'Total' ✅
Column 'wait_time' does not contain 'Total' ✅
Column 'wait_satisfaction' does not contain 'Total' ✅
Column 'mental_health_status' does not contain 'Total' ✅
Column 'needed_mental_health_care' does not contain 'Total' ✅
Column 'mental_healthcare_needs_met' does not contain 'Total' ✅
Column 'prescription_12mo' does not contain 'Total' ✅
Column 'cost_related_nonadherence' does not contain 'Total' ✅
Column 'unable_to_fill_prescription' does not contain 'Total' ✅
Column 'traveled_for_care' does not contain 'Total' ✅
Column 'reported_discrimination' does not contain 'Total' ✅
Column 'importance_indigenous_support' does not contain 'Total' ✅
Column 'reason_indigenous_support' does n

In [37]:
# drop any row where either column contains "Total"
hc_ft_df = hc_df[
    ~( # = drop
        hc_df['gender'].str.contains("Total") |
        hc_df['healthcare_access_experience'].str.contains("Total")
    )
]
print(hc_ft_df.shape)
hc_ft_df.head()

(253, 18)


,indigenous_group,gender,healthcare_access_experience,value,unmet_healthcare_need_past_12mo,consulted_provider_nonurgent_12mo,wait_time,wait_satisfaction,mental_health_status,needed_mental_health_care,mental_healthcare_needs_met,prescription_12mo,cost_related_nonadherence,unable_to_fill_prescription,traveled_for_care,reported_discrimination,importance_indigenous_support,reason_indigenous_support
10,First Nations,Men+,"Yes, had an unmet health care need in the past...",24.4,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,First Nations,Women+,"Yes, had an unmet health care need in the past...",38.6,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,Métis,Men+,"Yes, had an unmet health care need in the past...",25.5,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,Métis,Women+,"Yes, had an unmet health care need in the past...",33.9,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,Inuk (Inuit),Men+,"Yes, had an unmet health care need in the past...",25.7,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
contains_total(hc_ft_df)

Checking for 'Total' in columns:
Column 'indigenous_group' does not contain 'Total' ✅
Column 'gender' does not contain 'Total' ✅
Column 'healthcare_access_experience' does not contain 'Total' ✅
Column 'unmet_healthcare_need_past_12mo' does not contain 'Total' ✅
Column 'consulted_provider_nonurgent_12mo' does not contain 'Total' ✅
Column 'wait_time' does not contain 'Total' ✅
Column 'wait_satisfaction' does not contain 'Total' ✅
Column 'mental_health_status' does not contain 'Total' ✅
Column 'needed_mental_health_care' does not contain 'Total' ✅
Column 'mental_healthcare_needs_met' does not contain 'Total' ✅
Column 'prescription_12mo' does not contain 'Total' ✅
Column 'cost_related_nonadherence' does not contain 'Total' ✅
Column 'unable_to_fill_prescription' does not contain 'Total' ✅
Column 'traveled_for_care' does not contain 'Total' ✅
Column 'reported_discrimination' does not contain 'Total' ✅
Column 'importance_indigenous_support' does not contain 'Total' ✅
Column 'reason_indigenous

In [58]:
print(hc_ft_df.shape)

(253, 18)


> Healthcare Access 2017

In [8]:
hc_2017_df = pd.read_csv("data/csv_files/healthcare-access/healthcare_access_2017.csv")
hc_2017_df.head()

,geo,aboriginal_identity,age_group,sex,statistics,value,regular_medical_doctor_status,contact_with_health_professional_12mo,received_dental_care_3yr,healthcare_required_not_received_12mo
0,Canada,"Total, Aboriginal identity","Total, 15 years and over",Both sexes,Number of persons,998520.0,"Total, regular medical doctor status",NaN,NaN,NaN
1,Canada,"Total, Aboriginal identity","Total, 15 years and over",Both sexes,Percent,100.0,"Total, regular medical doctor status",NaN,NaN,NaN
2,Canada,"Total, Aboriginal identity","Total, 15 years and over",Both sexes,Number of persons,794220.0,Has a regular medical doctor,NaN,NaN,NaN
3,Canada,"Total, Aboriginal identity","Total, 15 years and over",Both sexes,Percent,79.5,Has a regular medical doctor,NaN,NaN,NaN
4,Canada,"Total, Aboriginal identity","Total, 15 years and over",Both sexes,Number of persons,196190.0,Does not have a regular medical doctor,NaN,NaN,NaN


In [32]:
# drop any row where either column contains "Total"
hc_2017_ft_df = hc_2017_df[
    ~( # = drop
        hc_2017_df['aboriginal_identity'].str.contains("Total") |
        hc_2017_df['age_group'].str.contains("Total") |
        hc_2017_df['regular_medical_doctor_status'].str.contains("Total") |
        hc_2017_df['contact_with_health_professional_12mo'].str.contains("Total") |
        hc_2017_df['received_dental_care_3yr'].str.contains("Total") |
        hc_2017_df['healthcare_required_not_received_12mo'].str.contains("Total")
    )
]
print(hc_2017_ft_df.shape)
hc_2017_ft_df.head()

(21319, 10)


,geo,aboriginal_identity,age_group,sex,statistics,value,regular_medical_doctor_status,contact_with_health_professional_12mo,received_dental_care_3yr,healthcare_required_not_received_12mo
1096,Canada,First Nations (North American Indian),15 to 24 years,Both sexes,Number of persons,85730.0,Has a regular medical doctor,NaN,NaN,NaN
1097,Canada,First Nations (North American Indian),15 to 24 years,Both sexes,Percent,73.5,Has a regular medical doctor,NaN,NaN,NaN
1098,Canada,First Nations (North American Indian),15 to 24 years,Both sexes,Number of persons,29410.0,Does not have a regular medical doctor,NaN,NaN,NaN
1099,Canada,First Nations (North American Indian),15 to 24 years,Both sexes,Percent,25.2,Does not have a regular medical doctor,NaN,NaN,NaN
1100,Canada,First Nations (North American Indian),15 to 24 years,Both sexes,Number of persons,2950.0,Does not have a regular medical doctor: no med...,NaN,NaN,NaN


In [39]:
contains_total(hc_2017_ft_df)

Checking for 'Total' in columns:
Column 'geo' does not contain 'Total' ✅
Column 'aboriginal_identity' does not contain 'Total' ✅
Column 'age_group' does not contain 'Total' ✅
Column 'sex' does not contain 'Total' ✅
Column 'statistics' does not contain 'Total' ✅
Column 'regular_medical_doctor_status' does not contain 'Total' ✅
Column 'contact_with_health_professional_12mo' does not contain 'Total' ✅
Column 'received_dental_care_3yr' does not contain 'Total' ✅
Column 'healthcare_required_not_received_12mo' does not contain 'Total' ✅


## Health Indicators Datasets

> can health indicator health provider

In [46]:
can_hi_pvd_df = pd.read_csv("data/csv_files/health-indicators/can_health_indicator_health_provider.csv")
can_hi_pvd_df.head()

,ref_date,geo,age_group,gender,unit,value,status,Belonging,Immunization,Life_Stress,...,Chronic_Condition,Healthcare_Provider,Perceived_Mental_Health,Alcohol_Drug_Use,Vaping,Perceived_Health,Physical_Activity,BMI_Adult,Breastfeeding,BMI_Youth
0,2018,Canada (excluding territories),12 to 17 years,Both sexes,Number of persons,13500.0,E,NaN,NaN,NaN,...,Arthritis (15 years and over),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018,Canada (excluding territories),12 to 17 years,Both sexes,Percent,1.3,E,NaN,NaN,NaN,...,Arthritis (15 years and over),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018,Canada (excluding territories),12 to 17 years,Both sexes,Statistically different from previous referenc...,0.0,E,NaN,NaN,NaN,...,Arthritis (15 years and over),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018,Canada (excluding territories),12 to 17 years,Both sexes,Number of persons,12000.0,E,NaN,NaN,NaN,...,Diabetes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018,Canada (excluding territories),12 to 17 years,Both sexes,Percent,0.5,E,NaN,NaN,NaN,...,Diabetes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
can_hi_pvd_ft_df = drop_total_rows(can_hi_pvd_df)
print(can_hi_pvd_df.shape, can_hi_pvd_ft_df.shape)
can_hi_pvd_ft_df.head()

(16854, 23) (15510, 23)


> Canadian Health Indicators

In [48]:
can_hi_df = pd.read_csv("data/csv_files/health-indicators/can_health_indicator.csv")
can_hi_df.head()

,ref_date,geo,age_group,gender,indicators,unit,value,status,BMI,Immunization,Breastfeeding,Life_stress,Perceived_mental_health,Perceived_health,Chronic_condition,Alcohol_drug_use,Smoking
0,2018,Canada (excluding territories),"Total, 18 years and over",Both sexes,"Perceived health, very good or excellent",Number_Thousands,17374.6,NaN,NaN,NaN,NaN,NaN,NaN,very good or excellent,NaN,NaN,NaN
1,2018,Canada (excluding territories),"Total, 18 years and over",Both sexes,"Perceived health, very good or excellent",Percent,59.7,NaN,NaN,NaN,NaN,NaN,NaN,very good or excellent,NaN,NaN,NaN
2,2018,Canada (excluding territories),"Total, 18 years and over",Both sexes,"Perceived health, fair or poor",Number_Thousands,3387.0,NaN,NaN,NaN,NaN,NaN,NaN,fair or poor,NaN,NaN,NaN
3,2018,Canada (excluding territories),"Total, 18 years and over",Both sexes,"Perceived health, fair or poor",Percent,11.6,NaN,NaN,NaN,NaN,NaN,NaN,fair or poor,NaN,NaN,NaN
4,2018,Canada (excluding territories),"Total, 18 years and over",Both sexes,"Perceived mental health, very good or excellent",Number_Thousands,19256.0,NaN,NaN,NaN,NaN,NaN,very good or excellent,NaN,NaN,NaN,NaN


In [53]:
can_hi_ft_df = drop_total_rows(can_hi_df)
print(can_hi_df.shape, can_hi_ft_df.shape)
can_hi_ft_df.head()

(37827, 17) (29068, 17)


,ref_date,geo,age_group,gender,indicators,unit,value,status,BMI,Immunization,Breastfeeding,Life_stress,Perceived_mental_health,Perceived_health,Chronic_condition,Alcohol_drug_use,Smoking
122,2018,Canada (excluding territories),18 to 34 years,Both sexes,"Perceived health, very good or excellent",Number_Thousands,5634.0,NaN,NaN,NaN,NaN,NaN,NaN,very good or excellent,NaN,NaN,NaN
123,2018,Canada (excluding territories),18 to 34 years,Both sexes,"Perceived health, very good or excellent",Percent,68.2,NaN,NaN,NaN,NaN,NaN,NaN,very good or excellent,NaN,NaN,NaN
124,2018,Canada (excluding territories),18 to 34 years,Both sexes,"Perceived health, fair or poor",Number_Thousands,573.3,NaN,NaN,NaN,NaN,NaN,NaN,fair or poor,NaN,NaN,NaN
125,2018,Canada (excluding territories),18 to 34 years,Both sexes,"Perceived health, fair or poor",Percent,6.9,NaN,NaN,NaN,NaN,NaN,NaN,fair or poor,NaN,NaN,NaN
126,2018,Canada (excluding territories),18 to 34 years,Both sexes,"Perceived mental health, very good or excellent",Number_Thousands,5192.0,NaN,NaN,NaN,NaN,NaN,very good or excellent,NaN,NaN,NaN,NaN


> Indigenous Health Indicators 2024

In [49]:
hi_24_df = pd.read_csv("data/csv_files/health-indicators/health_indicator24.csv")
hi_24_df.head()

,ref_date,geo,age_group,sex,indigenous_identity,indicators,characteristics,value
0,2015/2018,Canada,"Total, 18 years and over",Both sexes,"Total, Indigenous identity","Perceived health, very good or excellent",Percent,49.6
1,2015/2018,Canada,"Total, 18 years and over",Both sexes,"Total, Indigenous identity","Perceived health, good",Percent,32.1
2,2015/2018,Canada,"Total, 18 years and over",Both sexes,"Total, Indigenous identity","Perceived health, fair or poor",Percent,18.3
3,2015/2018,Canada,"Total, 18 years and over",Both sexes,"Total, Indigenous identity","Perceived mental health, very good or excellent",Percent,59.4
4,2015/2018,Canada,"Total, 18 years and over",Both sexes,"Total, Indigenous identity","Perceived mental health, good",Percent,27.8


In [54]:
hi_24_ft_df = drop_total_rows(hi_24_df)
print(hi_24_df.shape, hi_24_ft_df.shape)
hi_24_ft_df.head()

(36970, 8) (19447, 8)


,ref_date,geo,age_group,sex,indigenous_identity,indicators,characteristics,value
415,2015/2018,Canada,18 to 34 years,Both sexes,First Nations (North American Indian),"Perceived health, very good or excellent",Percent,55.8
416,2015/2018,Canada,18 to 34 years,Both sexes,First Nations (North American Indian),"Perceived health, good",Percent,33.4
417,2015/2018,Canada,18 to 34 years,Both sexes,First Nations (North American Indian),"Perceived health, fair or poor",Percent,10.8
418,2015/2018,Canada,18 to 34 years,Both sexes,First Nations (North American Indian),"Perceived mental health, very good or excellent",Percent,55.7
419,2015/2018,Canada,18 to 34 years,Both sexes,First Nations (North American Indian),"Perceived mental health, good",Percent,30.8


# Save Final Cleaned Data

In [61]:
# healthcare access
can_hc_ft_df.to_csv("data/cleaned_csv_files/healthcare-access/can_healthcare_access_cleaned.csv", index=False)
hc_ft_df.to_csv("data/cleaned_csv_files/healthcare-access/healthcare_access_cleaned.csv", index=False)
hc_2017_ft_df.to_csv("data/cleaned_csv_files/healthcare-access/healthcare_access_2017_cleaned.csv", index=False)

# health indicators
can_hi_pvd_ft_df.to_csv("data/cleaned_csv_files/health-indicators/can_health_indicator_health_provider_cleaned.csv", index=False)
can_hi_ft_df.to_csv("data/cleaned_csv_files/health-indicators/can_health_indicator_cleaned.csv", index=False)
hi_24_ft_df.to_csv("data/cleaned_csv_files/health-indicators/health_indicator24_cleaned.csv", index=False)
